In [ ]:
import numpy as np
import pandas as pd
import scipy.stats
import os
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib import rcParams

config = {
            "font.family": 'serif',
            "font.size": 12,# 相当于小四大小
            "mathtext.fontset": 'stix',#matplotlib渲染数学字体时使用的字体，和Times New Roman差别不大
            "font.serif": ['Arial'],#['Times New Roman'],#宋体
            'axes.unicode_minus': False # 处理负号，即-号
         }
rcParams.update(config)

# 模拟结果和实测数据是否一致的假设检验

In [14]:
def get_data(model_id,data,file):

    td = np.load('./'+file+'/databygroup.npy',allow_pickle=True).tolist()
    for k in td.keys():
        if model_id == 'MLP':
            data['yt'][k]=td[k][:,[-5,-4,-3,-2,-1]]
        elif model_id == 'LSTM' or model_id == 'BLSTM_ATT':
            n=td[k].shape[0]
            data['yt'][k]=td[k][:n-20-1,[-5,-4,-3,-2,-1]]
        else:
            n=td[k].shape[0]
            data['yt'][k]=td[k][:n-20-1,[-5,-4,-3,-2,-1]]

    for f in td.keys():
        data['yp'][f]=pd.read_csv('./'+file+'/results/'+model_id+'/results_'+f).values[:,1:]

def get_mse(data,file):
    # 不是每一类数据都有MSE，而是每次预测都有MSE
    datamaxmin=np.load('./'+file+'/databygroup_maxmin.npy',allow_pickle=True).tolist()
    datamax,datamin = datamaxmin['max'],datamaxmin['min']
    tem = []
    for k in data['yp'].keys():
        for i in range(data['yp'][k].shape[0]):
            datay = data['yp'][k][i]
            datayT = data['yt'][k][i]
            tem.append(np.sqrt(np.mean(np.square(datay-datayT))))
        
    data['mse']=tem

In [ ]:
model_id='LSTM'
alldataLSTM={'yp':{},
         'yt':{},
         'mse':{}}
get_data(model_id,alldataLSTM,'Step4_DNN-alldata')

augdataLSTM={'yp':{},
         'yt':{},
         'mse':{}}
get_data(model_id,augdataLSTM,'Step4_DNN-augdata')

scendataLSTM={'yp':{},
         'yt':{},
         'mse':{}}
get_data(model_id,scendataLSTM,'Step4_DNN-scenariodata')

model_id='BLSTM_ATT'
alldataATT={'yp':{},
         'yt':{},
         'mse':{}}
get_data(model_id,alldataATT,'Step4_DNN-alldata')


augdataATT={'yp':{},
         'yt':{},
         'mse':{}}
get_data(model_id,augdataATT,'Step4_DNN-augdata')

scendataATT={'yp':{},
         'yt':{},
         'mse':{}}
get_data(model_id,scendataATT,'Step4_DNN-scenariodata')

model_id='XGBoost'
alldataXGB={'yp':{},
         'yt':{},
         'mse':{}}
get_data(model_id,alldataXGB,'Step4_DNN-alldata')


augdataXGB={'yp':{},
         'yt':{},
         'mse':{}}
get_data(model_id,augdataXGB,'Step4_DNN-augdata')

scendataXGB={'yp':{},
         'yt':{},
         'mse':{}}
get_data(model_id,scendataXGB,'Step4_DNN-scenariodata')

get_mse(augdataLSTM,'Step4_DNN-augdata')
get_mse(alldataLSTM,'Step4_DNN-alldata')
get_mse(scendataLSTM,'Step4_DNN-scenariodata')
get_mse(augdataATT,'Step4_DNN-augdata')
get_mse(alldataATT,'Step4_DNN-alldata')
get_mse(scendataATT,'Step4_DNN-scenariodata')
get_mse(augdataXGB,'Step4_DNN-augdata')
get_mse(alldataXGB,'Step4_DNN-alldata')
get_mse(scendataXGB,'Step4_DNN-scenariodata')

In [37]:
# LSTM
lstm_true, lstm_alldata, lstm_scendata, lstm_augdata = [], [], [], []
for k in alldataLSTM['yt'].keys():
    for it in alldataLSTM['yt'][k]:
        lstm_true.append(it.tolist())
    for it in alldataLSTM['yp'][k]:
        lstm_alldata.append(it.tolist())
    for it in augdataLSTM['yp'][k]:
        lstm_augdata.append(it.tolist())
    for it in scendataLSTM['yp'][k]:
        lstm_scendata.append(it.tolist())

lstm_true = np.array(lstm_true)[3230:,:]
lstm_alldata = np.array(lstm_alldata)[3230:,:]
lstm_scendata = np.array(lstm_scendata)[3230:,:]
lstm_augdata = np.array(lstm_augdata)[3230:,:]
print('lstm:')
print(lstm_true.shape, lstm_alldata.shape, lstm_scendata.shape, lstm_augdata.shape)


# ATT
att_true, att_alldata, att_scendata, att_augdata = [], [], [], []
for k in alldataATT['yt'].keys():
    for it in alldataATT['yt'][k]:
        att_true.append(it.tolist())
    for it in alldataATT['yp'][k]:
        att_alldata.append(it.tolist())
    for it in augdataATT['yp'][k]:
        att_augdata.append(it.tolist())
    for it in scendataATT['yp'][k]:
        att_scendata.append(it.tolist())

att_true = np.array(att_true)[3230:,:]
att_alldata = np.array(att_alldata)[3230:,:]
att_scendata = np.array(att_scendata)[3230:,:]
att_augdata = np.array(att_augdata)[3230:,:]
print('att:')
print(att_true.shape, att_alldata.shape, att_scendata.shape, att_augdata.shape)


# XGB
xgb_true, xgb_alldata, xgb_scendata, xgb_augdata = [], [], [], []
for k in alldataXGB['yt'].keys():
    for it in alldataXGB['yt'][k]:
        xgb_true.append(it.tolist())
    for it in alldataXGB['yp'][k]:
        xgb_alldata.append(it.tolist())
    for it in augdataXGB['yp'][k]:
        xgb_augdata.append(it.tolist())
    for it in scendataXGB['yp'][k]:
        xgb_scendata.append(it.tolist())

xgb_true = np.array(xgb_true)[3230:,:]
xgb_alldata = np.array(xgb_alldata)[3230:,:]
xgb_scendata = np.array(xgb_scendata)[3230:,:]
xgb_augdata = np.array(xgb_augdata)[3230:,:]
print('xgb:')
print(xgb_true.shape, xgb_alldata.shape, xgb_scendata.shape, xgb_augdata.shape)

lstm:
(40554, 5) (40554, 5) (40554, 5) (40554, 5)
att:
(40554, 5) (40554, 5) (40554, 5) (40554, 5)
xgb:
(40554, 5) (40554, 5) (40554, 5) (40554, 5)


# 对误差分布的KS检验

In [41]:
def fre_count(mse):
    # 根据上述统计，最大MSE没有超过0.9的，用这个做范围
    rang_mse = np.arange(0,0.9,0.05)

    fre = [0 for _ in range(rang_mse.shape[0])]
    for it in mse:
        
        for i in range(rang_mse.shape[0]-1):
            if it > rang_mse[i] and it < rang_mse[i+1]:
                fre[i] += 1

    return np.array(fre)/np.sum(fre),rang_mse

In [42]:
FDLSTM_p1, FDLSTM_t = fre_count(alldataLSTM['mse'][3230:])
SCLSTM_p1, SCLSTM_t = fre_count(scendataLSTM['mse'][3230:])
KDLSTM_p1, KDLSTM_t = fre_count(augdataLSTM['mse'][3230:])

FDATT_p1, FDATT_t = fre_count(alldataATT['mse'][3230:])
SCATT_p1, SCATT_t = fre_count(scendataATT['mse'][3230:])
KDATT_p1, KDATT_t = fre_count(augdataATT['mse'][3230:])

FDXGB_p1, FDXGB_t = fre_count(alldataXGB['mse'][3230:])
SCXGB_p1, SCXGB_t = fre_count(scendataXGB['mse'][3230:])
KDXGB_p1, KDXGB_t = fre_count(augdataXGB['mse'][3230:])

In [51]:
r11 = scipy.stats.ks_2samp(FDLSTM_p1, KDLSTM_p1)
r12 = scipy.stats.ks_2samp(FDLSTM_p1, SCLSTM_p1)
r21 = scipy.stats.ks_2samp(FDATT_p1, KDATT_p1)
r22 = scipy.stats.ks_2samp(FDATT_p1, SCATT_p1)
r31 = scipy.stats.ks_2samp(FDXGB_p1, KDXGB_p1)
r32 = scipy.stats.ks_2samp(FDXGB_p1, SCXGB_p1)

In [ ]:
# 导出一张表
table = [
    [r11.statistic,r11.pvalue,r21.statistic,r21.pvalue,r31.statistic,r31.pvalue],
    [r12.statistic,r12.pvalue,r22.statistic,r22.pvalue,r32.statistic,r32.pvalue],
]
pd.DataFrame(table).to_csv('Table_predict_kstest.csv')